# nuPlan Dataset

In [14]:
import os
NUPLAN_DATA_ROOT = os.getenv('NUPLAN_DATA_ROOT', 'home/sgiron/workspace/data/nuplan/dataset')

In [15]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from nuplan.database.nuplan_db.db_cli_queries import (
    _get_table_row_count_from_db, 
    get_unique_frame_count,
    get_unique_frames_by_scenario_type
)

from tqdm import tqdm

def process_db_file(db_file):
    try:
        scenario_frame_count = get_unique_frames_by_scenario_type(db_file)
        tagged_frame_count = get_unique_frame_count(db_file)
        frame_count = _get_table_row_count_from_db(db_file, 'lidar_pc')
        return scenario_frame_count, tagged_frame_count, frame_count
    except Exception as e:
        return db_file, e

def get_scenario_count(db_files, workers=20):
    scenario_frames = defaultdict(int)
    total_tagged_frame_count = 0
    total_frame_count = 0

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(process_db_file, db_file) for db_file in db_files]
        for future in tqdm(futures, total=len(futures), desc="Processing database files"):
            scenario_frame_count, tagged_frame_count, frame_count = future.result()

            for scenario_type, count in scenario_frame_count:
                scenario_frames[scenario_type] += count

            total_tagged_frame_count += tagged_frame_count
            total_frame_count += frame_count

    return scenario_frames, total_tagged_frame_count, total_frame_count

# Mini Dataset

In [16]:
split = "mini"
nuplan_split_dir = f"{NUPLAN_DATA_ROOT}/nuplan-v1.1/splits/{split}"

db_files = []
for file in os.listdir(nuplan_split_dir):
    if file.endswith('.db'):
        db_files.append(os.path.join(nuplan_split_dir, file))

scenario_frames, total_tagged_frame_count, total_frame_count = get_scenario_count(db_files)

Processing database files:   0%|          | 0/64 [00:00<?, ?it/s]

Processing database files: 100%|██████████| 64/64 [00:00<00:00, 195.68it/s]


In [17]:
import pandas as pd


df = pd.DataFrame({
    'scenario_tag': list(scenario_frames.keys()), 
    'scenario_count': list(scenario_frames.values())
})

df.sort_values(by='scenario_count', ascending=False, inplace=True)

## Frames and Scenario Tags

Below is the total frame count, and the total tagged frames (frames having a scenario tag). A frame can have multiple scenario tags A frame can have multiple scenario tags.

### Frame Count

In [18]:
print(f"{total_frame_count:,}")

518,999


### Tagged Frames

In [19]:
scenario_tag_count = df['scenario_count'].sum()
print(f"{total_tagged_frame_count:,}")

390,186


### Scenario Tags

Here we see that the sum of unique frames for each tag is greater than the total frame count, so the relationship between frames and scenario-tags must be one-to-many.

In [20]:
total_scenario_tags = df['scenario_count'].sum()
print(f"{total_scenario_tags:,}")

821,831


Scenario Distribution

In [21]:
df['percentage_of_total_frames'] = ((df['scenario_count'] / total_frame_count) * 100).round(2)
df['percentage_of_scenario_frames'] = ((df['scenario_count'] / total_tagged_frame_count) * 100).round(2)
df

,scenario_tag,scenario_count,percentage_of_total_frames,percentage_of_scenario_frames
1,stationary,188367,36.29,48.28
5,on_intersection,84376,16.26,21.62
0,on_pickup_dropoff,78646,15.15,20.16
4,traversing_intersection,57786,11.13,14.81
7,on_traffic_light_intersection,57415,11.06,14.71
...,...,...,...,...
39,changing_lane_to_left,15,0.00,0.00
58,changing_lane_to_right,7,0.00,0.00
62,high_magnitude_jerk,7,0.00,0.00
51,behind_bike,2,0.00,0.00


In [22]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Table(
        header=dict(
            values=[
                "Scenario Tag", 
                "Scenario Count", 
                "Percentage of Total Frames", 
                "Percentage of Frames with a Scenario Tag"
            ]
        ),
        cells=dict(
            values=[
                df['scenario_tag'], 
                df['scenario_count'], 
                df['percentage_of_total_frames'],
                df['percentage_of_scenario_frames']
            ]
        )
    )
)

fig.update_layout(
    title='Scenario Tags',
    height=800,
    showlegend=False,
)
fig.show()

In [24]:
df['cumulative_percentage_of_total_scenario_frames'] = 100 * df['scenario_count'].cumsum() / df['scenario_count'].sum()

df_top = df.head(30)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_top['scenario_tag'],
    y=df_top['scenario_count'],
    name='Scenario Tag Count'
))
fig.add_trace(go.Scatter(
    x=df_top['scenario_tag'],
    y=df_top['cumulative_percentage_of_total_scenario_frames'],
    name='Cumulative %',
    yaxis='y2',
    line=dict(color='red', width=2)
))
fig.update_layout(
    title='Pareto Chart of Scenario Tags (Top 30)',
    yaxis=dict(title='Count'),
    yaxis2=dict(
        title='Cumulative %',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    legend=dict(x=0.7, y=1.1),
)
fig.show()

In [25]:
df_middle = df.iloc[20:40]
fig = go.Figure()
fig.add_trace(go.Bar(x=df_middle['scenario_tag'], y=df_middle['scenario_count'], name='Middle 20 Scenario Count'))
fig.update_layout(
    title='Frame Count for Middle 20 Scenarios', 
    xaxis_title='Scenario Tag', 
    yaxis_title='Percentage of Total Frames'
)
fig.show()

In [26]:
df_bottom = df.iloc[40:]
fig = go.Figure()
fig.add_trace(go.Bar(x=df_bottom['scenario_tag'], y=df_bottom['scenario_count'], name='Bottom 20 Scenario Count'))
fig.update_layout(
    title='Frame Count for Bottom 20 Scenarios', 
    xaxis_title='Scenario Tag', 
    yaxis_title='Percentage of Total Frames'
)
fig.show()